# The Perceptron — A Single Artificial Neuron Learns

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Notebook Guide (connected to Lab 57: The Anchor)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/appendix-2.ipynb)

**What you will do:** change the perceptron's learning rate and the size of its random starting weights to see how much of the final decision boundary is shaped by where training began.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Simulation

**Concept:** A perceptron is the simplest possible neural network—one artificial neuron that learns to classify data by adjusting its connection weights. Watch it learn in real time and see how it mirrors the basic logic of synaptic plasticity.

**Relevant Labs:** Lab 01 (prediction and error correction), Lab 57 (the pull of a starting point)

In [ ]:
# Perceptron Learning Visualisation

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Generate linearly separable data (two clusters)
n_points = 100
class_0 = np.random.randn(n_points // 2, 2) * 0.8 + np.array([-1.5, -1.5])
class_1 = np.random.randn(n_points // 2, 2) * 0.8 + np.array([1.5, 1.5])

X = np.vstack([class_0, class_1])
y = np.array([0] * (n_points // 2) + [1] * (n_points // 2))

# Add bias term
X_bias = np.c_[np.ones(n_points), X]

# Perceptron
weights = np.random.randn(3) * 0.1  # small random initial weights
learning_rate = 0.1
errors_per_epoch = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

epochs_to_show = [0, 1, 2, 5, 10, 20]
epoch = 0
max_epochs = 21

while epoch < max_epochs:
    errors = 0
    for i in range(n_points):
        prediction = 1 if np.dot(weights, X_bias[i]) >= 0 else 0
        error = y[i] - prediction
        if error != 0:
            weights += learning_rate * error * X_bias[i]
            errors += 1
    errors_per_epoch.append(errors)

    if epoch in epochs_to_show:
        idx = epochs_to_show.index(epoch)
        ax = axes[idx // 3, idx % 3]

        ax.scatter(class_0[:, 0], class_0[:, 1], c='salmon', s=30,
                   edgecolors='darkred', linewidth=0.5, label='Class 0')
        ax.scatter(class_1[:, 0], class_1[:, 1], c='steelblue', s=30,
                   edgecolors='darkblue', linewidth=0.5, label='Class 1')

        # Decision boundary: w0 + w1*x1 + w2*x2 = 0
        # => x2 = -(w0 + w1*x1) / w2
        if abs(weights[2]) > 1e-6:
            x_line = np.linspace(-4, 4, 100)
            y_line = -(weights[0] + weights[1] * x_line) / weights[2]
            ax.plot(x_line, y_line, 'k-', linewidth=2)

        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_title(f'Epoch {epoch} — {errors} errors', fontsize=12)
        ax.set_aspect('equal')
        if idx == 0:
            ax.legend(fontsize=9)

    epoch += 1

plt.suptitle('A Single Neuron Learns: The Perceptron Algorithm',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('perceptron_learning.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: the perceptron learns by prediction error—exactly "
      "like the brain. It makes a guess, checks the answer, and "
      "adjusts its weights (synaptic strengths) to reduce the error "
      "next time.")
print(f"After 20 epochs, the perceptron makes "
      f"{errors_per_epoch[-1]} errors on {n_points} data points.")

## Change the parameters

**weight_scale** sets how large the perceptron's random starting weights are -- its
"anchor" before any learning happens. **learning_rate** sets how big a step each mistake
produces. **seed** reshuffles both the two data clusters and the starting weights
together. The left panel shows the boundary right at the start; the right panel shows it
after training, reusing the same update rule as the cell above.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def explore(learning_rate=0.1, weight_scale=0.1, max_epochs=20, seed=42):
    np.random.seed(seed)
    n_points = 100
    class_0 = np.random.randn(n_points // 2, 2) * 0.8 + np.array([-1.5, -1.5])
    class_1 = np.random.randn(n_points // 2, 2) * 0.8 + np.array([1.5, 1.5])
    X = np.vstack([class_0, class_1])
    y = np.array([0] * (n_points // 2) + [1] * (n_points // 2))
    X_bias = np.c_[np.ones(n_points), X]

    weights = np.random.randn(3) * weight_scale
    initial_weights = weights.copy()
    errors_per_epoch = []

    for epoch in range(max_epochs):
        errors = 0
        for i in range(n_points):
            prediction = 1 if np.dot(weights, X_bias[i]) >= 0 else 0
            error = y[i] - prediction
            if error != 0:
                weights += learning_rate * error * X_bias[i]
                errors += 1
        errors_per_epoch.append(errors)

    def plot_boundary(ax, w, title):
        ax.scatter(class_0[:, 0], class_0[:, 1], c="salmon", s=25, edgecolors="darkred", linewidth=0.5)
        ax.scatter(class_1[:, 0], class_1[:, 1], c="steelblue", s=25, edgecolors="darkblue", linewidth=0.5)
        if abs(w[2]) > 1e-6:
            x_line = np.linspace(-4, 4, 100)
            y_line = -(w[0] + w[1] * x_line) / w[2]
            ax.plot(x_line, y_line, "k-", linewidth=2)
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=11)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    plot_boundary(ax1, initial_weights, f"Starting point (weight_scale={weight_scale})")
    plot_boundary(ax2, weights, f"After {max_epochs} epochs ({errors_per_epoch[-1]} errors)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.plot(errors_per_epoch, "o-", color="steelblue")
    plt.xlabel("Epoch")
    plt.ylabel("Errors")
    plt.title(f"Learning curve (learning_rate={learning_rate})")
    plt.show()

interact(explore,
         learning_rate=FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01),
         weight_scale=FloatSlider(value=0.1, min=0.0, max=2.0, step=0.1),
         max_epochs=IntSlider(value=20, min=1, max=50, step=1),
         seed=IntSlider(value=42, min=0, max=100, step=1));

## Questions to think about

1. Set weight_scale to 0, so training starts from exactly zero weights with no random "anchor" at all. Does the perceptron still find a good boundary? Now try weight_scale=2.0 with the same seed -- does a larger random starting point change how many epochs it takes to reach zero errors, even though the two classes have not changed?
2. Push learning_rate up to 1.0. Does the final boundary still look similar across a few different seeds, or does it become more erratic? What is the tradeoff of using a very large learning rate?
3. Set learning_rate very low, such as 0.01, and max_epochs to only 5. Does the perceptron have time to fully separate the two classes? How is this similar to the anchoring effect in Lab 57, where "the adjustment process terminates as soon as a plausible answer is reached" rather than continuing all the way to the best possible answer?
4. Try several different seed values (which reshuffles both the data and the starting weights together) with learning_rate and weight_scale fixed. Does the final decision boundary always end up in about the same place, or does the specific random starting point still leave a visible trace in the final answer?

## Challenge

Lab 57 shows that human numerical estimates are pulled toward an arbitrary starting anchor
and never fully correct for it. Test whether the perceptron shows an analogous residual
pull from its initial weights, by training it from many different random starting points on
the *same* fixed data and comparing how much the final decision boundary varies.

In [ ]:
def train_and_get_angle(seed, weight_scale=0.5, learning_rate=0.1, max_epochs=20):
    np.random.seed(0)  # keep the SAME two classes across every run
    n_points = 100
    class_0 = np.random.randn(n_points // 2, 2) * 0.8 + np.array([-1.5, -1.5])
    class_1 = np.random.randn(n_points // 2, 2) * 0.8 + np.array([1.5, 1.5])
    X = np.vstack([class_0, class_1])
    y = np.array([0] * (n_points // 2) + [1] * (n_points // 2))
    X_bias = np.c_[np.ones(n_points), X]

    rng = np.random.RandomState(seed)   # only the STARTING WEIGHTS vary across runs
    weights = rng.randn(3) * weight_scale
    for epoch in range(max_epochs):
        for i in range(n_points):
            prediction = 1 if np.dot(weights, X_bias[i]) >= 0 else 0
            error = y[i] - prediction
            if error != 0:
                weights += learning_rate * error * X_bias[i]
    return np.degrees(np.arctan2(-weights[1], weights[2]))  # boundary angle in degrees

angles = [train_and_get_angle(seed) for seed in range(20)]

plt.figure(figsize=(6, 3.5))
plt.hist(angles, bins=10, color="steelblue", edgecolor="white")
plt.xlabel("Final decision-boundary angle (degrees)")
plt.ylabel("Number of random starting points")
plt.title("Does the starting point leave a trace in the final answer?")
plt.show()
print(f"Spread across 20 starting points: {np.std(angles):.1f} degrees standard deviation.")
print("A small spread would mean the perceptron fully corrects for its starting point;")
print("a large spread would mean an anchoring-like residual pull remains.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "appendix-2.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")